<a href="https://colab.research.google.com/github/aisha13dikko-sudo/using-synthetic-data-for-thermal-comfort-classification/blob/main/wk_11_participant_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# wk11_participant_audit.ipynb
# Purpose: settle which participants report Cold, and reconcile the three
# different Cold prevalence figures currently circulating in the thesis.

!pip install -q datasets

import re, os
import pandas as pd
from datasets import load_dataset

os.makedirs("figure_data", exist_ok=True)

# Loading and ID extraction copied verbatim from wk10 cell 5, deliberately
# duplicated rather than imported so this notebook stands alone.
dataset  = load_dataset("kopetri/AutoTherm", "indoor")
train_df = dataset["train"].to_pandas()

def extract_participant_id(filename):
    m = re.search(r"participant_\d+", filename)
    return m.group() if m else "unknown"

train_df["participant_id"] = train_df["file_name"].apply(extract_participant_id)

# All 16 participants, not the 13-participant training pool. The disputed
# claim is about the whole cohort.
summary = (train_df.groupby("participant_id")["Label"]
           .agg(total_rows="size",
                cold_rows=lambda s: (s == -3).sum(),
                cold_pct=lambda s: round((s == -3).mean() * 100, 2))
           .sort_values("cold_pct", ascending=False))

print(summary.to_string())
print()
print("Participants in dataset      :", train_df["participant_id"].nunique())
print("Participants reporting Cold  :", int((summary["cold_rows"] > 0).sum()))
print("Cold-reporting IDs           :", list(summary[summary["cold_rows"] > 0].index))
print()

# Reconcile the three prevalence figures appearing in the thesis.
TEST_PARTICIPANTS = ["participant_14", "participant_16", "participant_20"]
pool = train_df[~train_df["participant_id"].isin(TEST_PARTICIPANTS)]
print(f"Dataset-wide Cold proportion  : {100*(train_df['Label']==-3).mean():.2f}%")
print(f"Training-pool Cold proportion : {100*(pool['Label']==-3).mean():.2f}%")

summary.to_csv("figure_data/cold_reporting_by_participant.csv")
print("\nSaved: figure_data/cold_reporting_by_participant.csv")

README.md:   0%|          | 0.00/8.57k [00:00<?, ?B/s]

indoor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 29.8MB            

indoor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

indoor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.41MB            

indoor/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1566728 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/194829 [00:00<?, ? examples/s]

                total_rows  cold_rows  cold_pct
participant_id                                 
participant_16       94677      22556     23.82
participant_2        94687      21505     22.71
participant_19      101163      16383     16.19
participant_11      101164      10132     10.02
participant_15       96220          0      0.00
participant_14       95568          0      0.00
participant_17      104246          0      0.00
participant_10       98429          0      0.00
participant_18      101344          0      0.00
participant_20       99774          0      0.00
participant_21      100151          0      0.00
participant_3        95000          0      0.00
participant_4        96094          0      0.00
participant_6        93353          0      0.00
participant_7        98540          0      0.00
participant_8        96318          0      0.00

Participants in dataset      : 16
Participants reporting Cold  : 4
Cold-reporting IDs           : ['participant_16', 'participant_2', '

In [2]:
from google.colab import files
files.download('figure_data/cold_reporting_by_participant.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>